
# 28A — V5 Sealed CONFIRM80 One-Shot Final Evaluation

This is the **final confirmatory evaluator** for the already-frozen V5 candidate.

It intentionally performs the remaining steps in one uninterrupted `Run All`:

1. verify 27A / candidate / Control / engine lineage,
2. hard-QA all 4 blind CONFIRM event-research batches,
3. freeze the complete event corpus and hashes **before pairing**,
4. deterministically build PRIMARY_TRUSTED and BROAD_SENSITIVITY 1–5y local pairs,
5. apply the frozen evidence-sufficiency gate,
6. **only if sufficient**, generate TG10 features and score the frozen V5 candidate + current Production Control once,
7. apply the already-predeclared 27A confirmation gates,
8. emit `PASS`, `INCONCLUSIVE`, or `FAIL`.

### Non-negotiable
- no event edits,
- no subject replacement,
- no pair selection,
- no feature changes,
- no coefficient changes,
- no threshold changes,
- no post-CONFIRM retuning.

If PRIMARY has fewer than 30 pairable subjects, the notebook does **not score V5 or Control**.
That preserves the option to collect a new independent confirmation wave without peeking.


In [1]:

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import hashlib, json, re, sys, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION="SAJU_ML_V5_CONFIRM_FINAL_ONE_SHOT_28A_20260817"
SEED=20260817
DATASET_PRIMARY="PRIMARY_TRUSTED"
DATASET_BROAD="BROAD_SENSITIVITY"

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repository (saju_engine.py required).")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def bool_series(s):
    if pd.api.types.is_bool_dtype(s):
        return s.astype(bool)
    m=s.astype(str).str.strip().str.lower().map({
        "true":True,"false":False,"1":True,"0":False
    })
    if m.isna().any():
        raise ValueError("Unparseable boolean values: "+repr(sorted(set(s[m.isna()].astype(str)))))
    return m.astype(bool)

ROOT=repo_root()

# 27A / sealed roster
FINAL_DIR=ROOT/"research/ml/artifacts/v5_final_roster"
CONFIRM_ROSTER=FINAL_DIR/"V5_CONFIRM_SUBJECT_ROSTER_80_SEALED.csv"
FINAL_ROSTER_FREEZE=FINAL_DIR/"V5_FINAL_ROSTER_FREEZE_DECISION.json"

COLL=ROOT/"research/ml/artifacts/v5_confirm_event_collection"
RESULTS=COLL/"results"
WORKLIST_FREEZE=COLL/"V5_CONFIRM_EVENT_WORKLIST_FREEZE_DECISION.json"
SUMMARY=COLL/"V5_CONFIRM_BLIND_EVENT_RESEARCH_BATCH_SUMMARY.csv"

# Frozen evaluation protocol and prior frozen candidate/control
PROTOCOL=ROOT/"research/ml_corpus/v5_ground_truth/V5_CONFIRM_ONE_SHOT_EVALUATION_PROTOCOL.json"
CAND_DIR=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced"
CAND_SPEC=CAND_DIR/"V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json"
CAND_COEF=CAND_DIR/"V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv"
CONTROL_DEC=ROOT/"research/ml/artifacts/v5_candidate_vs_control/V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION.json"

# Frozen exact-birth source
BIRTH_SNAPSHOT=ROOT/"research/ml/artifacts/v4_unified_dev_roster/PersonList-15k.csv"

OUT=ROOT/"research/ml/artifacts/v5_confirm_final_evaluation"
OUT.mkdir(parents=True,exist_ok=True)

required=[
    CONFIRM_ROSTER,FINAL_ROSTER_FREEZE,WORKLIST_FREEZE,SUMMARY,
    PROTOCOL,CAND_SPEC,CAND_COEF,CONTROL_DEC,BIRTH_SNAPSHOT
]
for b in range(1,5):
    required += [
        COLL/f"V5_CONFIRM_BATCH_{b:02d}_SUBJECT_WORKLIST.csv",
        RESULTS/f"batch_{b:02d}"/f"V5_CONFIRM_BATCH_{b:02d}_EVENTS.csv",
        RESULTS/f"batch_{b:02d}"/f"V5_CONFIRM_BATCH_{b:02d}_SUBJECT_SWEEP_AUDIT.csv",
        RESULTS/f"batch_{b:02d}"/f"V5_CONFIRM_BATCH_{b:02d}_RESEARCH_MANIFEST.json",
    ]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)

protocol=json.load(open(PROTOCOL,encoding="utf-8"))
cand_spec=json.load(open(CAND_SPEC,encoding="utf-8"))
control_dec=json.load(open(CONTROL_DEC,encoding="utf-8"))
roster_freeze=json.load(open(FINAL_ROSTER_FREEZE,encoding="utf-8"))
worklist_freeze=json.load(open(WORKLIST_FREEZE,encoding="utf-8"))

assert protocol["status"]=="PREDECLARED_AFTER_DEV_CANDIDATE_AND_CONTROL_GATE_BEFORE_CONFIRM_ROSTER_OPEN"
assert protocol["entry_requirements"]["required_26A_status"]=="V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL"
assert protocol["entry_requirements"]["required_candidate"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
assert int(protocol["evidence_sufficiency"]["PRIMARY_min_pairable_subjects"])==30
assert int(protocol["evaluation"]["bootstrap_iterations"])==20000

assert cand_spec["status"]=="FROZEN_BEFORE_CONTROL_AND_CONFIRM"
assert cand_spec["architecture"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
assert cand_spec["axis_required_at_inference"] is False
assert cand_spec["coefficients_sha256"]==sha256_file(CAND_COEF)

assert control_dec["status"]=="V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL"
assert control_dec["confirm_protocol_may_begin"] is True
assert control_dec["candidate"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"

# Critical implementation freeze: current engine must be exactly the same file benchmarked in 26A.
expected_engine_sha=control_dec["lineage"]["saju_engine_py_sha256"]
current_engine_sha=sha256_file(ROOT/"saju_engine.py")
assert current_engine_sha==expected_engine_sha, (
    "saju_engine.py changed since 26A. Restore the exact 26A engine before CONFIRM scoring. "
    f"expected={expected_engine_sha} current={current_engine_sha}"
)

assert worklist_freeze["status"]=="V5_CONFIRM_EVENT_WORKLIST_FROZEN_READY_FOR_BLIND_EVENT_COLLECTION"
assert worklist_freeze["confirm_roster_sha256"]==sha256_file(CONFIRM_ROSTER)
assert worklist_freeze["candidate_coefficients_sha256"]==sha256_file(CAND_COEF)
assert worklist_freeze["confirm_evaluation_protocol_sha256"]==sha256_file(PROTOCOL)

assert int(roster_freeze["confirm_n"])==80
assert roster_freeze["confirm_roster_sha256"]==sha256_file(CONFIRM_ROSTER)

print("PRE-FINAL LINEAGE QA: PASS")
print("Candidate:",cand_spec["architecture"])
print("Engine SHA locked:",current_engine_sha)
print("27A protocol SHA:",sha256_file(PROTOCOL))


PRE-FINAL LINEAGE QA: PASS
Candidate: TG10_MULTITASK_EFFECT_RIDGE_BALANCED
Engine SHA locked: d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e
27A protocol SHA: 55a3340c5cf9e52eeffec5708ded805997f5922ad515a1d5cbf86ca1141a0ee9


## 1. Hard-QA all 4 blind event-research batches

In [2]:

confirm=pd.read_csv(CONFIRM_ROSTER)
confirm["subject_id"]=confirm["subject_id"].astype(str)
assert len(confirm)==80 and confirm.subject_id.nunique()==80
assert confirm.preassigned_axis.value_counts().to_dict()=={
    "STATUS":35,"PROJECT":25,"COMPETITIVE":20
}

all_events=[]
all_audits=[]
batch_lineage=[]

for b in range(1,5):
    tag=f"{b:02d}"
    work_path=COLL/f"V5_CONFIRM_BATCH_{tag}_SUBJECT_WORKLIST.csv"
    event_path=RESULTS/f"batch_{tag}"/f"V5_CONFIRM_BATCH_{tag}_EVENTS.csv"
    audit_path=RESULTS/f"batch_{tag}"/f"V5_CONFIRM_BATCH_{tag}_SUBJECT_SWEEP_AUDIT.csv"
    manifest_path=RESULTS/f"batch_{tag}"/f"V5_CONFIRM_BATCH_{tag}_RESEARCH_MANIFEST.json"

    work=pd.read_csv(work_path)
    ev=pd.read_csv(event_path)
    audit=pd.read_csv(audit_path)
    man=json.load(open(manifest_path,encoding="utf-8"))

    for z in [work,ev,audit]:
        z["subject_id"]=z["subject_id"].astype(str)

    assert man["status"]==f"CONFIRM_BATCH_{tag}_FIRST_PASS_SOURCE_SWEEP_COMPLETE_NOT_EVENT_FREEZE"
    assert man["roster_kind"]=="SEALED_V5_CONFIRM80"
    assert int(man["batch_id"])==b
    assert man["rules"]["blind_event_research"] is True
    assert man["rules"]["no_astrology_used"] is True
    assert man["rules"]["no_control_used"] is True
    assert man["rules"]["no_pairability_used"] is True
    assert man["rules"]["no_pair_gap_inspected"] is True
    assert man["rules"]["preassigned_axis_immutable"] is True
    assert man["rules"]["one_sided_subjects_not_backfilled"] is True

    assert man["lineage"]["worklist_sha256"]==sha256_file(work_path)
    assert man["lineage"]["events_sha256"]==sha256_file(event_path)
    assert man["lineage"]["audit_sha256"]==sha256_file(audit_path)

    assert len(work)==int(man["n_subjects"])
    assert work.subject_id.nunique()==len(work)
    assert set(audit.subject_id)==set(work.subject_id)
    assert audit.subject_id.nunique()==len(work)

    # Exact roster membership and axis lineage.
    chk=work[["subject_id","name","preassigned_axis"]].merge(
        confirm[["subject_id","name","preassigned_axis"]],
        on="subject_id",how="left",suffixes=("_work","_roster"),validate="one_to_one"
    )
    assert chk["name_roster"].notna().all()
    assert (chk["name_work"].astype(str)==chk["name_roster"].astype(str)).all()
    assert (chk["preassigned_axis_work"].astype(str)==chk["preassigned_axis_roster"].astype(str)).all()

    assert set(ev.subject_id).issubset(set(work.subject_id))
    em=ev[["subject_id","name","preassigned_axis"]].merge(
        work[["subject_id","name","preassigned_axis"]],
        on="subject_id",how="left",suffixes=("_event","_work"),validate="many_to_one"
    )
    assert em["name_work"].notna().all()
    assert (em["name_event"].astype(str)==em["name_work"].astype(str)).all()
    assert (em["preassigned_axis_event"].astype(str)==em["preassigned_axis_work"].astype(str)).all()

    ev["exclude"]=bool_series(ev["exclude"])
    ev["event_year"]=pd.to_numeric(ev["event_year"],errors="raise").astype(int)
    ev["source_quality_norm"]=ev["source_quality"].astype(str).str.strip().str.lower()

    assert ev.polarity.isin(["positive","negative"]).all()
    assert ev.event_year.between(1900,2026).all()
    assert ev.source_quality_norm.isin(["high","medium"]).all()
    assert ev.source_url.astype(str).str.match(r"^https?://").all()
    assert ev.source_title.fillna("").astype(str).str.strip().ne("").all()
    assert ev.source_publisher.fillna("").astype(str).str.strip().ne("").all()
    assert ev.event_type.fillna("").astype(str).str.strip().ne("").all()
    assert ev.event_description.fillna("").astype(str).str.strip().ne("").all()
    assert ev.loc[ev.exclude,"exclude_reason"].fillna("").astype(str).str.strip().ne("").all()

    eligible=ev[~ev.exclude].copy()
    assert len(eligible)==int(man["eligible_rows"])
    assert int(ev.exclude.sum())==int(man["excluded_audit_rows"])

    # Audit counts must reproduce from event rows.
    calc=[]
    for sid in work.subject_id:
        es=eligible[eligible.subject_id==sid]
        calc.append({
            "subject_id":sid,
            "eligible_event_rows_calc":len(es),
            "positive_eligible_calc":int((es.polarity=="positive").sum()),
            "negative_eligible_calc":int((es.polarity=="negative").sum()),
        })
    calc=pd.DataFrame(calc)
    ac=audit.merge(calc,on="subject_id",how="left",validate="one_to_one")
    assert (ac.eligible_event_rows.astype(int)==ac.eligible_event_rows_calc.astype(int)).all()
    assert (ac.positive_eligible.astype(int)==ac.positive_eligible_calc.astype(int)).all()
    assert (ac.negative_eligible.astype(int)==ac.negative_eligible_calc.astype(int)).all()

    ev["batch_id"]=b
    audit["batch_id"]=b
    all_events.append(ev)
    all_audits.append(audit)
    batch_lineage.append({
        "batch":tag,
        "worklist_sha256":sha256_file(work_path),
        "events_sha256":sha256_file(event_path),
        "audit_sha256":sha256_file(audit_path),
        "manifest_sha256":sha256_file(manifest_path),
        "subjects":len(work),
        "event_rows":len(ev),
        "eligible_rows":len(eligible)
    })

events=pd.concat(all_events,ignore_index=True,sort=False)
audits=pd.concat(all_audits,ignore_index=True,sort=False)

assert audits.subject_id.nunique()==80
assert len(audits)==80

# Ensure all four frozen worklists cover the sealed roster exactly once.
all_work=pd.concat([
    pd.read_csv(COLL/f"V5_CONFIRM_BATCH_{b:02d}_SUBJECT_WORKLIST.csv")
    for b in range(1,5)
],ignore_index=True)
all_work["subject_id"]=all_work["subject_id"].astype(str)
assert len(all_work)==80 and all_work.subject_id.nunique()==80
assert set(all_work.subject_id)==set(confirm.subject_id)

# No exact semantic duplicate among research-eligible rows.
research_eligible=events[~events.exclude].copy()
dup_key=["subject_id","preassigned_axis","event_year","polarity","event_type","event_description"]
dups=research_eligible[research_eligible.duplicated(dup_key,keep=False)]
assert len(dups)==0,f"Exact semantic duplicate eligible events: {len(dups)}"

# Cross-check non-authoritative batch summary for accidental file-placement/mixing errors.
summary=pd.read_csv(SUMMARY)
assert int(summary.subjects.sum())==80
assert int(summary.event_rows.sum())==len(events)
assert int(summary.eligible_rows.sum())==len(research_eligible)
assert int(summary.excluded_rows.sum())==int(events.exclude.sum())

print("CONFIRM BLIND EVENT QA: PASS")
print("subjects:",audits.subject_id.nunique())
print("full rows:",len(events))
print("research eligible:",len(research_eligible))
print("excluded audit rows:",int(events.exclude.sum()))


CONFIRM BLIND EVENT QA: PASS
subjects: 80
full rows: 158
research eligible: 149
excluded audit rows: 9


## 2. Freeze the complete CONFIRM event corpus before pairing

In [3]:

# Canonical deterministic order; no factual edits.
sort_cols=["batch_id","research_order_in_batch","subject_id","event_year","polarity","event_type","event_description"]
sort_cols=[c for c in sort_cols if c in events.columns]
events_frozen=events.sort_values(sort_cols).reset_index(drop=True)

FULL_EVENT=OUT/"V5_CONFIRM_EVENT_CORPUS_FULL_FROZEN.csv"
ELIG_EVENT=OUT/"V5_CONFIRM_EVENT_CORPUS_RESEARCH_ELIGIBLE_FROZEN.csv"
events_frozen.to_csv(FULL_EVENT,index=False)
events_frozen[~events_frozen.exclude].to_csv(ELIG_EVENT,index=False)

freeze_decision={
    "version":"V5_CONFIRM_EVENT_CORPUS_FREEZE_DECISION_V1",
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_CONFIRM_EVENT_CORPUS_FROZEN_READY_FOR_DETERMINISTIC_PAIRING",
    "confirm_subjects":80,
    "full_event_rows":int(len(events_frozen)),
    "research_eligible_rows":int((~events_frozen.exclude).sum()),
    "excluded_audit_rows":int(events_frozen.exclude.sum()),
    "full_event_sha256":sha256_file(FULL_EVENT),
    "eligible_event_sha256":sha256_file(ELIG_EVENT),
    "sealed_roster_sha256":sha256_file(CONFIRM_ROSTER),
    "worklist_freeze_sha256":sha256_file(WORKLIST_FREEZE),
    "confirm_protocol_sha256":sha256_file(PROTOCOL),
    "batch_lineage":batch_lineage,
    "rules":{
        "membership_changed":False,
        "axis_changed":False,
        "event_year_or_polarity_changed":False,
        "event_rows_deleted_for_pairability":False,
        "pairability_used_before_event_freeze":False,
        "astrology_used_before_event_freeze":False,
        "control_used_before_event_freeze":False,
        "network_refetch_in_this_notebook":False,
        "source_basis":"blind research manifests + frozen source fields + schema/lineage QA"
    }
}
FREEZE_DEC=OUT/"V5_CONFIRM_EVENT_FREEZE_DECISION.json"
json.dump(freeze_decision,open(FREEZE_DEC,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps({
    "status":freeze_decision["status"],
    "full_event_rows":freeze_decision["full_event_rows"],
    "eligible_event_rows":freeze_decision["research_eligible_rows"],
    "full_event_sha256":freeze_decision["full_event_sha256"]
},ensure_ascii=False,indent=2))


{
  "status": "V5_CONFIRM_EVENT_CORPUS_FROZEN_READY_FOR_DETERMINISTIC_PAIRING",
  "full_event_rows": 158,
  "eligible_event_rows": 149,
  "full_event_sha256": "3089d6be06d79784824acf02c08f43730795d0571790141d48117f35e46b691c"
}


## 3. Create frozen PRIMARY/BROAD confidence sets and deterministic 1–5y pairs

In [4]:

FUZZY_PRIMARY_TOKENS=list(
    protocol["event_freeze_and_confidence_sets"]["PRIMARY_TRUSTED"]["exclude_event_type_tokens"]
)

def fuzzy_primary(event_type):
    s=str(event_type).casefold()
    return any(tok in s for tok in FUZZY_PRIMARY_TOKENS)

eligible=events_frozen[~events_frozen.exclude].copy()
eligible["source_quality_norm"]=eligible.source_quality.astype(str).str.strip().str.lower()
eligible["fuzzy_primary_semantics"]=eligible.event_type.map(fuzzy_primary)

primary_events=eligible[
    (eligible.source_quality_norm=="high")
    & (~eligible.fuzzy_primary_semantics)
].copy()

broad_events=eligible[
    eligible.source_quality_norm.isin(["high","medium"])
].copy()

PRIMARY_EVENT=OUT/"V5_CONFIRM_EVENT_MODEL_PRIMARY_TRUSTED.csv"
BROAD_EVENT=OUT/"V5_CONFIRM_EVENT_MODEL_BROAD_SENSITIVITY.csv"
primary_events.to_csv(PRIMARY_EVENT,index=False)
broad_events.to_csv(BROAD_EVENT,index=False)

roster=confirm.copy()
roster["birth_date"]=pd.to_datetime(roster["birth_date"],errors="coerce")
assert roster.birth_date.notna().all()
roster["birth_year"]=roster.birth_date.dt.year.astype(int)

def normalize_events_for_pairing(frame):
    x=frame.copy()
    x["collection_wave"]="CONFIRM"
    return (
        x.groupby(
            ["subject_id","name","preassigned_axis","collection_wave","event_year","polarity"],
            as_index=False
        )
        .agg(
            event_count=("event_type","size"),
            event_types=("event_type",lambda s:" | ".join(sorted(set(map(str,s))))),
            source_urls=("source_url",lambda s:" | ".join(sorted(set(map(str,s)))))
        )
    )

def build_local_pairs(frame,dataset_name):
    ev=normalize_events_for_pairing(frame)
    birth=roster[["subject_id","birth_year"]].drop_duplicates()
    ev=ev.merge(birth,on="subject_id",how="left",validate="many_to_one")
    assert ev.birth_year.notna().all()

    out=[]
    for (sid,axis,wave),g in ev.groupby(
        ["subject_id","preassigned_axis","collection_wave"],sort=True
    ):
        pos=g[g.polarity=="positive"]
        neg=g[g.polarity=="negative"]
        for _,p in pos.iterrows():
            for _,n in neg.iterrows():
                gap=abs(int(p.event_year)-int(n.event_year))
                if not (1<=gap<=5):
                    continue
                if int(p.event_year)<int(n.event_year):
                    earlier,later=p,n
                    earlier_is_positive=1
                else:
                    earlier,later=n,p
                    earlier_is_positive=0
                out.append({
                    "subject_id":str(sid),
                    "name":str(earlier["name"]),
                    "preassigned_axis":str(axis),
                    "collection_wave":"CONFIRM",
                    "birth_year":int(earlier.birth_year),
                    "earlier_year":int(earlier.event_year),
                    "later_year":int(later.event_year),
                    "year_gap":int(gap),
                    "earlier_is_positive":int(earlier_is_positive),
                    "earlier_polarity":str(earlier.polarity),
                    "later_polarity":str(later.polarity),
                    "earlier_event_count":int(earlier.event_count),
                    "later_event_count":int(later.event_count),
                    "earlier_event_types":str(earlier.event_types),
                    "later_event_types":str(later.event_types),
                })

    pairs=pd.DataFrame(out)
    if not len(pairs):
        return pairs

    key=["subject_id","preassigned_axis","collection_wave","earlier_year","later_year","earlier_is_positive"]
    pairs=pairs.drop_duplicates(key).sort_values(
        ["subject_id","earlier_year","later_year","earlier_is_positive"]
    ).reset_index(drop=True)

    counts=pairs.groupby("subject_id").size().rename("subject_pair_count")
    pairs=pairs.merge(counts,on="subject_id",how="left")
    pairs["subject_weight"]=1.0/pairs.subject_pair_count
    pairs["earlier_event_age"]=pairs.earlier_year-pairs.birth_year
    pairs["later_event_age"]=pairs.later_year-pairs.birth_year
    pairs["calendar_midpoint"]=(pairs.earlier_year+pairs.later_year)/2.0

    pairs["positive_year"]=np.where(
        pairs.earlier_is_positive.eq(1),pairs.earlier_year,pairs.later_year
    ).astype(int)
    pairs["negative_year"]=np.where(
        pairs.earlier_is_positive.eq(1),pairs.later_year,pairs.earlier_year
    ).astype(int)
    pairs["positive_earlier_calc"]=(pairs.positive_year<pairs.negative_year).astype(int)
    pairs["positive_age"]=pairs.positive_year-pairs.birth_year
    pairs["negative_age"]=pairs.negative_year-pairs.birth_year
    pairs["abs_year_gap"]=(pairs.positive_year-pairs.negative_year).abs().astype(int)
    pairs["dataset"]=dataset_name
    pairs["pair_id"]=[f"CONFIRM_{dataset_name}_{i+1:04d}" for i in range(len(pairs))]
    return pairs

pairs_primary=build_local_pairs(primary_events,DATASET_PRIMARY)
pairs_broad=build_local_pairs(broad_events,DATASET_BROAD)

PRIMARY_PAIRS=OUT/"V5_CONFIRM_LOCAL_PAIRS_PRIMARY_TRUSTED.csv"
BROAD_PAIRS=OUT/"V5_CONFIRM_LOCAL_PAIRS_BROAD_SENSITIVITY.csv"
pairs_primary.to_csv(PRIMARY_PAIRS,index=False)
pairs_broad.to_csv(BROAD_PAIRS,index=False)

def coverage(frame,dataset):
    rows=[]
    groups=[("TOTAL",frame)]
    if len(frame):
        groups += [(a,g) for a,g in frame.groupby("preassigned_axis")]
    for key,g in groups:
        if len(g):
            subj=int(g.subject_id.nunique())
            pairn=int(len(g))
            posshare=float(g.positive_earlier_calc.mean())
            subshare=float(g.groupby("subject_id").positive_earlier_calc.mean().mean())
        else:
            subj=pairn=0
            posshare=subshare=np.nan
        rows.append({
            "dataset":dataset,"group":key,
            "pair_rows":pairn,"pairable_subjects":subj,
            "positive_earlier_pair_share":posshare,
            "positive_earlier_subject_macro_share":subshare
        })
    return rows

cov=pd.DataFrame(
    coverage(pairs_primary,DATASET_PRIMARY)
    + coverage(pairs_broad,DATASET_BROAD)
)
COVERAGE=OUT/"V5_CONFIRM_PAIR_COVERAGE.csv"
cov.to_csv(COVERAGE,index=False)
display(cov)

min_subjects=int(protocol["evidence_sufficiency"]["PRIMARY_min_pairable_subjects"])
primary_subjects=int(pairs_primary.subject_id.nunique()) if len(pairs_primary) else 0
orientation_classes=sorted(pairs_primary.positive_earlier_calc.unique().tolist()) if len(pairs_primary) else []

READY_TO_SCORE=(primary_subjects>=min_subjects and orientation_classes==[0,1])

if primary_subjects<min_subjects:
    insuff_reason=f"PRIMARY pairable subjects {primary_subjects} < frozen minimum {min_subjects}"
elif orientation_classes!=[0,1]:
    insuff_reason="Chronology-balanced primary metric undefined because both chronology orientations are not present."
else:
    insuff_reason=None

pair_decision={
    "version":"V5_CONFIRM_PAIRABILITY_DECISION_V1",
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":(
        "V5_CONFIRM_PAIR_SET_FROZEN_READY_FOR_ONE_SHOT_SCORING"
        if READY_TO_SCORE else
        "V5_CONFIRM_INSUFFICIENT_EVIDENCE_NO_PRODUCTION_DECISION"
    ),
    "primary_pair_rows":int(len(pairs_primary)),
    "primary_pairable_subjects":primary_subjects,
    "broad_pair_rows":int(len(pairs_broad)),
    "broad_pairable_subjects":int(pairs_broad.subject_id.nunique()) if len(pairs_broad) else 0,
    "primary_chronology_classes":orientation_classes,
    "frozen_min_pairable_subjects":min_subjects,
    "ready_to_score":bool(READY_TO_SCORE),
    "insufficiency_reason":insuff_reason,
    "primary_pairs_sha256":sha256_file(PRIMARY_PAIRS),
    "broad_pairs_sha256":sha256_file(BROAD_PAIRS),
    "event_freeze_sha256":sha256_file(FREEZE_DEC),
    "rules":{
        "event_corpus_was_frozen_before_pairing":True,
        "all_1_to_5y_pairs_used":True,
        "directional_pair_selection":False,
        "subject_replacement":False,
        "astrology_scored_yet":False,
        "control_scored_yet":False
    }
}
PAIR_DEC=OUT/"V5_CONFIRM_PAIRABILITY_DECISION.json"
json.dump(pair_decision,open(PAIR_DEC,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps({
    "status":pair_decision["status"],
    "primary_pairs":pair_decision["primary_pair_rows"],
    "primary_subjects":primary_subjects,
    "ready_to_score":READY_TO_SCORE,
    "reason":insuff_reason
},ensure_ascii=False,indent=2))


,dataset,group,pair_rows,pairable_subjects,positive_earlier_pair_share,positive_earlier_subject_macro_share
0,PRIMARY_TRUSTED,TOTAL,15,8,0.533333,0.656250
1,PRIMARY_TRUSTED,COMPETITIVE,14,7,0.500000,0.607143
2,PRIMARY_TRUSTED,STATUS,1,1,1.000000,1.000000
3,BROAD_SENSITIVITY,TOTAL,35,22,0.685714,0.768939
4,BROAD_SENSITIVITY,COMPETITIVE,24,13,0.666667,0.762821
5,BROAD_SENSITIVITY,PROJECT,2,2,1.000000,1.000000
6,BROAD_SENSITIVITY,STATUS,9,7,0.666667,0.714286


{
  "status": "V5_CONFIRM_INSUFFICIENT_EVIDENCE_NO_PRODUCTION_DECISION",
  "primary_pairs": 15,
  "primary_subjects": 8,
  "ready_to_score": false,
  "reason": "PRIMARY pairable subjects 8 < frozen minimum 30"
}


## 4. If sufficient: hydrate exact birth inputs and generate frozen TG10 + Control scores once

In [5]:

SCORING_PERFORMED=False

if not READY_TO_SCORE:
    print("SCORING SKIPPED BY FROZEN EVIDENCE-SUFFICIENCY GATE.")
else:
    sys.path.insert(0,str(ROOT))
    import saju_engine as se
    import sajupy

    # Re-assert engine implementation freeze immediately before scoring.
    assert sha256_file(ROOT/"saju_engine.py")==expected_engine_sha

    needed_subjects=set(pairs_broad.subject_id.astype(str))
    r=confirm[confirm.subject_id.astype(str).isin(needed_subjects)].copy()
    assert set(r.subject_id.astype(str))==needed_subjects

    snapshot=pd.read_csv(BIRTH_SNAPSHOT)
    required_snapshot_cols={"RowKey","BirthTime","Gender","Name","Notes"}
    assert required_snapshot_cols.issubset(snapshot.columns)

    def parse_birth_blob(blob):
        obj=json.loads(str(blob))
        std=obj["StdTime"]
        loc=obj["Location"]
        m=re.match(
            r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
            std.strip()
        )
        if not m:
            raise ValueError(std)
        hh,mm,dd,mo,yyyy,offset=m.groups()
        return {
            "snapshot_birth_date":"%04d-%02d-%02d"%(int(yyyy),int(mo),int(dd)),
            "snapshot_birth_time":"%02d:%02d"%(int(hh),int(mm)),
            "snapshot_utc_offset":offset,
            "snapshot_birth_place":loc["Name"],
            "snapshot_longitude":float(loc["Longitude"]),
            "snapshot_latitude":float(loc["Latitude"]),
        }

    snap_rows=[]
    for _,row in snapshot.iterrows():
        try:
            parsed=parse_birth_blob(row["BirthTime"])
        except Exception:
            continue
        snap_rows.append({"source_row_key":str(row["RowKey"]),**parsed})
    snap=pd.DataFrame(snap_rows).drop_duplicates("source_row_key")

    assert "source_row_key" in r.columns
    r["source_row_key"]=r.source_row_key.astype(str)
    r=r.merge(snap,on="source_row_key",how="left",validate="many_to_one")

    for target,source in [
        ("birth_date","snapshot_birth_date"),
        ("birth_time","snapshot_birth_time"),
        ("utc_offset","snapshot_utc_offset"),
        ("birth_place","snapshot_birth_place"),
        ("longitude","snapshot_longitude"),
        ("latitude","snapshot_latitude"),
    ]:
        if target not in r.columns:
            r[target]=np.nan
        missing=r[target].isna()|(r[target].astype(str).str.strip()=="")
        r.loc[missing,target]=r.loc[missing,source]

    assert r.snapshot_birth_date.notna().all()
    assert (
        pd.to_datetime(r.birth_date).dt.strftime("%Y-%m-%d")
        ==pd.to_datetime(r.snapshot_birth_date).dt.strftime("%Y-%m-%d")
    ).all(),"Roster birth date disagrees with frozen PersonList-15k snapshot."

    for c in ["birth_date","birth_time","utc_offset","longitude","latitude","gender"]:
        assert r[c].notna().all(),f"Missing exact engine input: {c}"
        assert ~(r[c].astype(str).str.strip()=="").any(),f"Blank exact engine input: {c}"

    ENGINE_INPUT=OUT/"V5_CONFIRM_ENGINE_INPUT_AUDIT.csv"
    r.to_csv(ENGINE_INPUT,index=False)

    STEMS=list("甲乙丙丁戊己庚辛壬癸")
    BRANCHES=list("子丑寅卯辰巳午未申酉戌亥")
    STEM_ELEMENT_EN={
        "甲":"wood","乙":"wood","丙":"fire","丁":"fire","戊":"earth",
        "己":"earth","庚":"metal","辛":"metal","壬":"water","癸":"water"
    }
    STEM_YINYANG={
        "甲":"yang","乙":"yin","丙":"yang","丁":"yin","戊":"yang",
        "己":"yin","庚":"yang","辛":"yin","壬":"yang","癸":"yin"
    }
    HIDDEN_STEMS={
        "子":["癸"],"丑":["己","癸","辛"],"寅":["甲","丙","戊"],"卯":["乙"],
        "辰":["戊","乙","癸"],"巳":["丙","戊","庚"],"午":["丁","己"],
        "未":["己","丁","乙"],"申":["庚","壬","戊"],"酉":["辛"],
        "戌":["戊","辛","丁"],"亥":["壬","甲"]
    }
    GENERATES={
        "wood":"fire","fire":"earth","earth":"metal","metal":"water","water":"wood"
    }
    CONTROLS={
        "wood":"earth","earth":"water","water":"fire","fire":"metal","metal":"wood"
    }
    EXACT_TENGODS=[
        "bijian","jiecai","shishen","shangguan",
        "pian_cai","zheng_cai","qi_sha","zheng_guan",
        "pian_yin","zheng_yin"
    ]

    def parse_utc_offset(raw):
        if isinstance(raw,(int,float)) and not pd.isna(raw):
            return float(raw)
        s=str(raw).strip()
        sign=-1.0 if s.startswith("-") else 1.0
        s=s[1:] if s[:1] in "+-" else s
        hh,mm=s.split(":")
        return sign*(int(hh)+int(mm)/60.0)

    def public_lon_corrected_birth(subject):
        y,m,d=map(int,str(subject["birth_date"])[:10].split("-"))
        hh,mi=map(int,str(subject["birth_time"]).split(":")[:2])
        lon=float(subject["longitude"])
        utc_hours=parse_utc_offset(subject["utc_offset"])
        calc=sajupy.get_saju_calculator()
        civil=datetime(y,m,d,hh,mi)
        corr=float(calc._calculate_solar_time_correction(lon,utc_hours))
        h2,min2,dc=calc._adjust_time_for_solar(civil.hour,civil.minute,corr)
        y2,m2,d2=calc._adjust_date_for_solar(civil.year,civil.month,civil.day,dc)
        return {"y":y2,"m":m2,"d":d2,"h":h2,"min":min2}

    def pillar_chars(value):
        if isinstance(value,(list,tuple)) and len(value)>=2:
            a,b=str(value[0]),str(value[1])
            if a in STEMS and b in BRANCHES:
                return a,b
        s=str(value)
        stem=next((c for c in s if c in STEMS),None)
        branch=next((c for c in s if c in BRANCHES),None)
        return stem,branch

    def sexagenary_year_pillar(year):
        i=(int(year)-1984)%60
        return STEMS[i%10]+BRANCHES[i%12]

    def get_daewoon_pillar(meta_row):
        for key in ("대운_pillar","대운","daewoon_pillar"):
            if meta_row.get(key):
                st,br=pillar_chars(meta_row[key])
                if st and br:
                    return st+br
        return None

    def get_sewoon_pillar(meta_row,year):
        for key in ("세운_pillar","세운","연주","year_pillar"):
            if meta_row.get(key):
                st,br=pillar_chars(meta_row[key])
                if st and br:
                    return st+br
        return sexagenary_year_pillar(year)

    def exact_ten_god(day_stem,other_stem):
        de=STEM_ELEMENT_EN[day_stem]
        oe=STEM_ELEMENT_EN[other_stem]
        same_polarity=STEM_YINYANG[day_stem]==STEM_YINYANG[other_stem]
        if de==oe:
            return "bijian" if same_polarity else "jiecai"
        if GENERATES[de]==oe:
            return "shishen" if same_polarity else "shangguan"
        if CONTROLS[de]==oe:
            return "pian_cai" if same_polarity else "zheng_cai"
        if CONTROLS[oe]==de:
            return "qi_sha" if same_polarity else "zheng_guan"
        if GENERATES[oe]==de:
            return "pian_yin" if same_polarity else "zheng_yin"
        raise RuntimeError((day_stem,other_stem))

    def exact_tg_distribution(day_stem,stems):
        counts=Counter(exact_ten_god(day_stem,st) for st in stems)
        n=float(max(1,len(stems)))
        return {tg:counts[tg]/n for tg in EXACT_TENGODS}

    def tg10_year_features(result,meta_row,year):
        natal=result["원국"]
        day_stem=pillar_chars(natal["day"])[0]
        dw_stem,dw_branch=pillar_chars(get_daewoon_pillar(meta_row))
        sw_stem,sw_branch=pillar_chars(get_sewoon_pillar(meta_row,year))
        if not all([day_stem,dw_stem,dw_branch,sw_stem,sw_branch]):
            raise ValueError(f"Pillar parse failed {year}")

        f={}
        for label,stem,branch in [("dw",dw_stem,dw_branch),("sw",sw_stem,sw_branch)]:
            tg=exact_ten_god(day_stem,stem)
            for name in EXACT_TENGODS:
                f[f"tg10__{label}_stem_{name}"]=float(tg==name)
            dist=exact_tg_distribution(day_stem,HIDDEN_STEMS[branch])
            for name in EXACT_TENGODS:
                f[f"tg10__{label}_branch_hidden_{name}"]=float(dist[name])
        return f

    def control_score(meta_row):
        candle=meta_row.get("candle") or {}
        v=candle.get("close")
        return float(v) if v is not None else np.nan

    calculator=sajupy.get_saju_calculator()
    min_year=int(calculator.min_year)
    max_year=int(calculator.max_year)

    needed_years=defaultdict(set)
    for frame in [pairs_primary,pairs_broad]:
        for _,row in frame.iterrows():
            needed_years[str(row.subject_id)].add(int(row.positive_year))
            needed_years[str(row.subject_id)].add(int(row.negative_year))

    subject_map={str(row.subject_id):row.to_dict() for _,row in r.iterrows()}
    assert set(needed_years).issubset(subject_map)

    year_rows=[]
    engine_failures=[]
    for i,sid in enumerate(sorted(needed_years)):
        subject=subject_map[sid]
        try:
            corrected=public_lon_corrected_birth(subject)
            if not (min_year<=int(corrected["y"])<=max_year):
                raise ValueError((sid,corrected["y"],min_year,max_year))
            inp=se.BirthInput(
                year=int(corrected["y"]),month=int(corrected["m"]),day=int(corrected["d"]),
                hour=int(corrected["h"]),minute=int(corrected["min"]),
                gender=str(subject["gender"]),calendar="solar",is_leap_month=False,
                use_solar_time=False,utc_offset=9
            )
            result=se.compute_all(inp)
            meta={int(row["year"]):row for row in result["chart_data"]["연도별_타임라인"]}
            for year in sorted(needed_years[sid]):
                if year not in meta:
                    raise RuntimeError(f"Timeline missing {sid} {year}")
                feats=tg10_year_features(result,meta[year],year)
                ctrl=control_score(meta[year])
                if not np.isfinite(ctrl):
                    raise RuntimeError(f"Non-finite Control score {sid} {year}")
                year_rows.append({
                    "subject_id":sid,"year":int(year),
                    "control_score":float(ctrl),**feats
                })
        except Exception as e:
            engine_failures.append({
                "subject_id":sid,"name":subject.get("name"),"error":repr(e)
            })
        if (i+1)%10==0:
            print("engine subjects:",i+1,"/",len(needed_years))

    FAIL_PATH=OUT/"V5_CONFIRM_ENGINE_FEATURE_FAILURES.csv"
    pd.DataFrame(engine_failures).to_csv(FAIL_PATH,index=False)
    if engine_failures:
        raise RuntimeError(
            f"CONFIRM engine/feature generation failed for {len(engine_failures)} subjects. "
            f"Do not delete subjects or guess birth times. See {FAIL_PATH}."
        )

    year_features=pd.DataFrame(year_rows)
    YEAR_FEATURE=OUT/"V5_CONFIRM_TG10_YEAR_FEATURES_AND_CONTROL.csv"
    year_features.to_csv(YEAR_FEATURE,index=False)

    coef=pd.read_csv(CAND_COEF)
    expected_features=list(cand_spec["features"])
    assert len(expected_features)==40
    assert coef.feature.tolist()==expected_features
    assert set(expected_features)==set(["diff__"+c for c in year_features.columns if c.startswith("tg10__")])

    lookup=year_features.set_index(["subject_id","year"])

    def score_pairs(frame,dataset):
        out=[]
        for _,row in frame.iterrows():
            sid=str(row.subject_id)
            p=lookup.loc[(sid,int(row.positive_year))]
            n=lookup.loc[(sid,int(row.negative_year))]
            raw=[]
            for feat in expected_features:
                primitive=feat[len("diff__"):]
                raw.append(float(p[primitive]-n[primitive]))
            raw=np.asarray(raw,float)
            mean=coef.scaler_mean.to_numpy(float)
            scale=coef.scaler_scale.to_numpy(float)
            beta=coef.global_coefficient.to_numpy(float)
            assert np.all(np.isfinite(raw))
            assert np.all(scale>0)
            standardized=(raw-mean)/scale
            cand_score=float(standardized@beta)

            cp=float(p.control_score)
            cn=float(n.control_score)
            cand_correct=1.0 if cand_score>1e-12 else (0.0 if cand_score<-1e-12 else 0.5)
            ctrl_correct=1.0 if cp>cn else (0.0 if cp<cn else 0.5)
            reverse_ctrl_correct=1.0-ctrl_correct if ctrl_correct in (0.0,1.0) else 0.5

            out.append({
                **row.to_dict(),
                "candidate_pair_score":cand_score,
                "candidate_correct":cand_correct,
                "control_positive_score":cp,
                "control_negative_score":cn,
                "control_correct":ctrl_correct,
                "control_reversed_correct":reverse_ctrl_correct,
                "candidate_coefficients_sha256":sha256_file(CAND_COEF),
                "engine_sha256":current_engine_sha
            })
        return pd.DataFrame(out)

    scored_primary=score_pairs(pairs_primary,DATASET_PRIMARY)
    scored_broad=score_pairs(pairs_broad,DATASET_BROAD)

    SCORED_PRIMARY=OUT/"V5_CONFIRM_PRIMARY_PAIR_SCORES.csv"
    SCORED_BROAD=OUT/"V5_CONFIRM_BROAD_PAIR_SCORES.csv"
    scored_primary.to_csv(SCORED_PRIMARY,index=False)
    scored_broad.to_csv(SCORED_BROAD,index=False)

    SCORING_PERFORMED=True
    print("ONE-SHOT V5 + CONTROL SCORING COMPLETE")


SCORING SKIPPED BY FROZEN EVIDENCE-SUFFICIENCY GATE.


## 5. Frozen metrics, subject bootstrap, and final PASS / INCONCLUSIVE / FAIL

In [6]:

FINAL_DEC=OUT/"V5_CONFIRM_FINAL_DECISION.json"

if not READY_TO_SCORE:
    final_decision={
        "version":"V5_CONFIRM_FINAL_DECISION_V1",
        "created_at":datetime.now().isoformat(timespec="seconds"),
        "status":"V5_CONFIRM_INSUFFICIENT_EVIDENCE_NO_PRODUCTION_DECISION",
        "classification":"INCONCLUSIVE",
        "reason":insuff_reason,
        "scoring_performed":False,
        "primary_pairable_subjects":primary_subjects,
        "frozen_min_pairable_subjects":min_subjects,
        "event_freeze_sha256":sha256_file(FREEZE_DEC),
        "primary_pairs_sha256":sha256_file(PRIMARY_PAIRS),
        "broad_pairs_sha256":sha256_file(BROAD_PAIRS),
        "rules":{
            "candidate_retuned":False,
            "control_retuned":False,
            "confirm_outcomes_used_for_model_fitting":False,
            "new_confirmation_wave_may_be_considered_without_having_seen_scores":True
        },
        "next_rule":protocol["if_inconclusive"]
    }
    json.dump(final_decision,open(FINAL_DEC,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    print(json.dumps(final_decision,ensure_ascii=False,indent=2))
else:
    assert SCORING_PERFORMED

    def chronology_balanced_metric(frame,col):
        vals=[]
        for ori in [0,1]:
            h=frame[frame.positive_earlier_calc==ori]
            if h.empty:
                return np.nan
            vals.append(h.groupby("subject_id")[col].mean().mean())
        return float(np.mean(vals))

    def orientation_metric(frame,col,ori):
        h=frame[frame.positive_earlier_calc==ori]
        return float(h.groupby("subject_id")[col].mean().mean()) if len(h) else np.nan

    def headline(frame,label):
        return {
            "dataset":label,
            "n_pairs":int(len(frame)),
            "n_subjects":int(frame.subject_id.nunique()),
            "candidate_balanced":chronology_balanced_metric(frame,"candidate_correct"),
            "control_balanced":chronology_balanced_metric(frame,"control_correct"),
            "control_reversed_balanced":chronology_balanced_metric(frame,"control_reversed_correct"),
            "delta_candidate_minus_control":(
                chronology_balanced_metric(frame,"candidate_correct")
                -chronology_balanced_metric(frame,"control_correct")
            ),
            "delta_candidate_minus_reversed_control":(
                chronology_balanced_metric(frame,"candidate_correct")
                -chronology_balanced_metric(frame,"control_reversed_correct")
            ),
            "candidate_positive_later":orientation_metric(frame,"candidate_correct",0),
            "candidate_positive_earlier":orientation_metric(frame,"candidate_correct",1),
            "control_positive_later":orientation_metric(frame,"control_correct",0),
            "control_positive_earlier":orientation_metric(frame,"control_correct",1),
        }

    h_primary=headline(scored_primary,DATASET_PRIMARY)
    h_broad=headline(scored_broad,DATASET_BROAD)
    HEADLINE=pd.DataFrame([h_primary,h_broad])
    HEADLINE_PATH=OUT/"V5_CONFIRM_HEADLINE.csv"
    HEADLINE.to_csv(HEADLINE_PATH,index=False)
    display(HEADLINE)

    # Paired subject bootstrap, preserving all pairs for each sampled subject.
    def subject_orientation_matrix(frame,col,subjects):
        ix={s:i for i,s in enumerate(subjects)}
        M=np.full((len(subjects),2),np.nan)
        z=frame.groupby(["subject_id","positive_earlier_calc"])[col].mean().reset_index()
        for _,row in z.iterrows():
            M[ix[str(row.subject_id)],int(row.positive_earlier_calc)]=float(row[col])
        return M

    def metric_counts(M,counts):
        vals=[]
        for ori in [0,1]:
            mask=~np.isnan(M[:,ori])
            den=counts[mask].sum()
            if den<=0:
                return np.nan
            vals.append(np.sum(M[mask,ori]*counts[mask])/den)
        return float(np.mean(vals))

    nboot=int(protocol["evaluation"]["bootstrap_iterations"])
    subjects=sorted(scored_primary.subject_id.astype(str).unique())
    C=subject_orientation_matrix(scored_primary,"candidate_correct",subjects)
    K=subject_orientation_matrix(scored_primary,"control_correct",subjects)

    rng=np.random.default_rng(SEED)
    cand_minus_chance=[]
    cand_minus_control=[]
    attempts=0
    while len(cand_minus_control)<nboot and attempts<nboot*5:
        attempts+=1
        sample=rng.integers(0,len(subjects),len(subjects))
        counts=np.bincount(sample,minlength=len(subjects))
        cm=metric_counts(C,counts)
        km=metric_counts(K,counts)
        if np.isnan(cm) or np.isnan(km):
            continue
        cand_minus_chance.append(cm-0.5)
        cand_minus_control.append(cm-km)

    if len(cand_minus_control)<nboot:
        raise RuntimeError(
            f"Could produce only {len(cand_minus_control)} valid chronology-balanced bootstrap "
            f"replicates of requested {nboot}."
        )

    dchance=np.asarray(cand_minus_chance)
    dcontrol=np.asarray(cand_minus_control)
    boot=pd.DataFrame([
        {
            "comparison":"FROZEN_V5_minus_CHANCE_0p5",
            "observed_delta":h_primary["candidate_balanced"]-0.5,
            "bootstrap_mean_delta":float(dchance.mean()),
            "ci025":float(np.quantile(dchance,.025)),
            "ci975":float(np.quantile(dchance,.975)),
            "p_delta_gt_0":float((dchance>0).mean()),
            "valid_iterations":len(dchance)
        },
        {
            "comparison":"FROZEN_V5_minus_PRODUCTION_CONTROL",
            "observed_delta":h_primary["delta_candidate_minus_control"],
            "bootstrap_mean_delta":float(dcontrol.mean()),
            "ci025":float(np.quantile(dcontrol,.025)),
            "ci975":float(np.quantile(dcontrol,.975)),
            "p_delta_gt_0":float((dcontrol>0).mean()),
            "valid_iterations":len(dcontrol)
        }
    ])
    BOOT_PATH=OUT/"V5_CONFIRM_BOOTSTRAP.csv"
    boot.to_csv(BOOT_PATH,index=False)
    display(boot)

    # Axis diagnostics only; never gate.
    axis_rows=[]
    for dataset,frame in [(DATASET_PRIMARY,scored_primary),(DATASET_BROAD,scored_broad)]:
        for axis,g in frame.groupby("preassigned_axis"):
            row=headline(g,f"{dataset}__{axis}")
            row["axis"]=axis
            axis_rows.append(row)
    axis=pd.DataFrame(axis_rows)
    AXIS_PATH=OUT/"V5_CONFIRM_AXIS_DIAGNOSTICS.csv"
    axis.to_csv(AXIS_PATH,index=False)
    display(axis)

    gates_cfg=protocol["confirmation_gates"]
    bchance=boot.set_index("comparison").loc["FROZEN_V5_minus_CHANCE_0p5"]
    bcontrol=boot.set_index("comparison").loc["FROZEN_V5_minus_PRODUCTION_CONTROL"]

    gates={
        "candidate_PRIMARY_balanced_ge_055":
            h_primary["candidate_balanced"]>=float(gates_cfg["candidate_PRIMARY_balanced_min"]),
        "candidate_positive_later_ge_050":
            h_primary["candidate_positive_later"]>=float(gates_cfg["candidate_positive_later_min"]),
        "candidate_positive_earlier_ge_050":
            h_primary["candidate_positive_earlier"]>=float(gates_cfg["candidate_positive_earlier_min"]),
        "candidate_minus_chance_bootstrap_lower95_gt_0":
            float(bchance.ci025)>float(gates_cfg["candidate_minus_chance_bootstrap_lower95_gt"]),
        "candidate_minus_Control_observed_delta_gt_0":
            h_primary["delta_candidate_minus_control"]>float(gates_cfg["candidate_minus_Control_observed_delta_gt"]),
        "candidate_minus_Control_bootstrap_lower95_gt_0":
            float(bcontrol.ci025)>float(gates_cfg["candidate_minus_Control_bootstrap_lower95_gt"]),
        "candidate_minus_Control_bootstrap_probability_gt0_ge_095":
            float(bcontrol.p_delta_gt_0)>=float(gates_cfg["candidate_minus_Control_bootstrap_probability_gt0_min"]),
        "BROAD_candidate_balanced_ge_052":
            h_broad["candidate_balanced"]>=float(gates_cfg["BROAD_candidate_balanced_min"]),
        "BROAD_candidate_minus_Control_delta_ge_0":
            h_broad["delta_candidate_minus_control"]>=float(gates_cfg["BROAD_candidate_minus_Control_delta_min"]),
    }

    all_gates=all(gates.values())

    if all_gates:
        classification="PASS"
        status="V5_CONFIRM_PASS_FROZEN_CANDIDATE_REPLICATES_AND_BEATS_CONTROL"
        next_rule=protocol["if_pass"]
    else:
        directionally_favorable=(
            h_primary["candidate_balanced"]>0.5
            and h_primary["delta_candidate_minus_control"]>0.0
        )
        if directionally_favorable:
            classification="INCONCLUSIVE"
            status="V5_CONFIRM_INCONCLUSIVE_FROZEN_CONFIDENCE_GATE_NOT_CLEARED"
            next_rule=protocol["if_inconclusive"]
        else:
            classification="FAIL"
            status="V5_CONFIRM_FAIL_REJECT_PRODUCTION_PROMOTION"
            next_rule=protocol["if_fail"]

    final_decision={
        "version":"V5_CONFIRM_FINAL_DECISION_V1",
        "notebook_version":NOTEBOOK_VERSION,
        "created_at":datetime.now().isoformat(timespec="seconds"),
        "status":status,
        "classification":classification,
        "candidate":"TG10_MULTITASK_EFFECT_RIDGE_BALANCED",
        "primary_metric":"chronology_balanced_subject_macro_pairwise_accuracy",
        "scoring_performed":True,
        "primary":h_primary,
        "broad":h_broad,
        "bootstrap":boot.to_dict(orient="records"),
        "gates":gates,
        "all_gates":bool(all_gates),
        "control_direction_sensitivity":{
            "official_direction":"higher_is_better",
            "primary_reversed_control_balanced":h_primary["control_reversed_balanced"],
            "primary_candidate_minus_reversed_control":h_primary["delta_candidate_minus_reversed_control"],
            "role":"report_only_not_gate"
        },
        "lineage":{
            "event_freeze_sha256":sha256_file(FREEZE_DEC),
            "primary_pairs_sha256":sha256_file(PRIMARY_PAIRS),
            "broad_pairs_sha256":sha256_file(BROAD_PAIRS),
            "candidate_spec_sha256":sha256_file(CAND_SPEC),
            "candidate_coefficients_sha256":sha256_file(CAND_COEF),
            "26A_control_decision_sha256":sha256_file(CONTROL_DEC),
            "confirm_protocol_sha256":sha256_file(PROTOCOL),
            "saju_engine_py_sha256":current_engine_sha
        },
        "rules":{
            "candidate_retuned_after_CONFIRM":False,
            "control_retuned_after_CONFIRM":False,
            "features_changed_after_CONFIRM":False,
            "thresholds_changed_after_CONFIRM":False,
            "pair_membership_changed_after_scoring":False,
            "CONFIRM_used_for_model_fitting":False,
            "axis_diagnostics_used_as_gate":False
        },
        "next_rule":next_rule
    }
    json.dump(final_decision,open(FINAL_DEC,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

    print(json.dumps({
        "status":status,
        "classification":classification,
        "primary_candidate":h_primary["candidate_balanced"],
        "primary_control":h_primary["control_balanced"],
        "delta":h_primary["delta_candidate_minus_control"],
        "chance_ci025":float(bchance.ci025),
        "control_delta_ci025":float(bcontrol.ci025),
        "p_candidate_gt_control":float(bcontrol.p_delta_gt_0),
        "all_gates":all_gates
    },ensure_ascii=False,indent=2))


{
  "version": "V5_CONFIRM_FINAL_DECISION_V1",
  "created_at": "2026-08-17T15:56:30",
  "status": "V5_CONFIRM_INSUFFICIENT_EVIDENCE_NO_PRODUCTION_DECISION",
  "classification": "INCONCLUSIVE",
  "reason": "PRIMARY pairable subjects 8 < frozen minimum 30",
  "scoring_performed": false,
  "primary_pairable_subjects": 8,
  "frozen_min_pairable_subjects": 30,
  "event_freeze_sha256": "7e905f9588efd966afe7adeeee1762dc14034dc2bf7820059d694921ce8b4405",
  "primary_pairs_sha256": "7d47969a8d3d8b576eacf3140f30c2d3bf3f0a998323f24fe0ed8ce1d601fc3f",
  "broad_pairs_sha256": "c1cbb2ddec25819c4f048b052087611755f8852cfa4561c77663c5d2000cfdb0",
  "rules": {
    "candidate_retuned": false,
    "control_retuned": false,
    "confirm_outcomes_used_for_model_fitting": false,
    "new_confirmation_wave_may_be_considered_without_having_seen_scores": true
  },
  "next_rule": "Do not tune on CONFIRM. Decide separately whether a new independently frozen confirmation cohort is worth collecting."
}



## What to send back

Always send:

```text
V5_CONFIRM_FINAL_DECISION.json
V5_CONFIRM_EVENT_FREEZE_DECISION.json
V5_CONFIRM_PAIR_COVERAGE.csv
V5_CONFIRM_PAIRABILITY_DECISION.json
```

If `scoring_performed = true`, also send:

```text
V5_CONFIRM_HEADLINE.csv
V5_CONFIRM_BOOTSTRAP.csv
V5_CONFIRM_AXIS_DIAGNOSTICS.csv
```

Do **not** change events, pairs, coefficients, thresholds, or engine code after seeing the result.
